# 2015 Main Training Pipeline (Clinical + Dietary)

This notebook is a clean 2015-only training pipeline with no experimentation branches.

Included stages:
1. Load and merge 2015 clinical + dietary datasets.
2. Preprocess features and create target.
3. Standardize and impute missing values.
4. Train core models.
5. Calibrate top models and select the best calibrated setup.

In [1]:
import json
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import KNNImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, log_loss
)

warnings.filterwarnings("ignore")

RANDOM_SEED = 2026
np.random.seed(RANDOM_SEED)

print(f"Random seed: {RANDOM_SEED}")

Random seed: 2026


In [2]:
PROJECT_ROOT_CANDIDATES = [
    Path.cwd(),
    Path.cwd().parent,
    Path(r"c:/Jon/College/Thesis"),
]

PROJECT_ROOT = next((p for p in PROJECT_ROOT_CANDIDATES if (p / "Datasets2015").exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate project root containing 'Datasets2015'.")

DATASETS_DIR = PROJECT_ROOT / "Datasets2015"
CLINICAL_DIR = DATASETS_DIR / "Clinical"
DIETARY_DIR = DATASETS_DIR / "Dietary"

ARTIFACT_DIR = PROJECT_ROOT / "V2.2.1.1" / "main_2015_training_artifacts"
MODEL_DIR = ARTIFACT_DIR / "models"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

def find_dataset_file(folder: Path, dataset_hint: str) -> Path:
    if not folder.exists():
        raise FileNotFoundError(f"Folder not found: {folder}")

    candidates = sorted([
        p for p in folder.glob("*.csv")
        if "data-set" in p.name.lower() and dataset_hint in p.name.lower()
    ])

    if not candidates:
        candidates = sorted([
            p for p in folder.glob("*.csv")
            if "data-set" in p.name.lower()
        ])

    if not candidates:
        raise FileNotFoundError(f"No data-set CSV found in: {folder}")

    return candidates[0]

clinical_path = find_dataset_file(CLINICAL_DIR, "clinical")
dietary_path = find_dataset_file(DIETARY_DIR, "dietary")

print(f"Clinical file: {clinical_path.name}")
print(f"Dietary file: {dietary_path.name}")
print(f"Artifacts: {ARTIFACT_DIR}")

Clinical file: Jonathan Ralph_Baes_2026-03-26141903_data-set_clinical.csv
Dietary file: Jonathan Ralph_Baes_2026-03-26141801_data-set_dietary.csv
Artifacts: c:\Jon\College\Thesis\V2.2.1.1\main_2015_training_artifacts


In [3]:
clinical_df = pd.read_csv(clinical_path, low_memory=False)
dietary_df = pd.read_csv(dietary_path, low_memory=False)

# Prefer the most specific key set available in both datasets.
merge_key_options = [
    ["enns_year", "hhnum", "member_code"],
    ["hhnum", "member_code"],
    ["enns_year", "hhnum"],
    ["hhnum"],
]

merge_keys = next(
    (
        keys
        for keys in merge_key_options
        if all(k in clinical_df.columns and k in dietary_df.columns for k in keys)
    ),
    [],
)

if not merge_keys:
    common_keys = sorted(set(clinical_df.columns).intersection(set(dietary_df.columns)))
    raise ValueError(
        "No usable common merge keys found between clinical and dietary datasets. "
        f"Common columns are: {common_keys}"
    )

dietary_work = dietary_df.copy()
dietary_dup_count = int(dietary_work.duplicated(subset=merge_keys).sum())
if dietary_dup_count > 0:
    print(
        f"Warning: dietary has {dietary_dup_count:,} duplicate rows on keys {merge_keys}. "
        "Keeping first occurrence per key to preserve clinical row count."
    )
    dietary_work = dietary_work.sort_values(by=merge_keys).drop_duplicates(subset=merge_keys, keep="first")

overlap_non_keys = [
    c for c in dietary_work.columns
    if c in clinical_df.columns and c not in merge_keys
]
dietary_trim = dietary_work.drop(columns=overlap_non_keys, errors="ignore")

merged_df = clinical_df.merge(dietary_trim, on=merge_keys, how="left")

print(f"Clinical shape: {clinical_df.shape}")
print(f"Dietary shape: {dietary_df.shape}")
print(f"Merge keys: {merge_keys}")
print(f"Overlapping non-key dietary columns dropped: {len(overlap_non_keys)}")
print(f"Merged shape: {merged_df.shape}")
print(f"Clinical rows preserved: {len(merged_df) == len(clinical_df)}")
if len(merged_df) != len(clinical_df):
    print(
        "Warning: merged row count differs from clinical row count. "
        "Check right-side key uniqueness assumptions."
    )

merged_out = ARTIFACT_DIR / "merged_2015_clinical_dietary.csv"
merged_df.to_csv(merged_out, index=False)
print(f"Saved merged dataset: {merged_out}")

Clinical shape: (151189, 23)
Dietary shape: (9925, 45)
Merge keys: ['hhnum']
Overlapping non-key dietary columns dropped: 4
Merged shape: (151189, 63)
Clinical rows preserved: True
Saved merged dataset: c:\Jon\College\Thesis\V2.2.1.1\main_2015_training_artifacts\merged_2015_clinical_dietary.csv


In [4]:
def find_bp_column(df: pd.DataFrame, kind: str) -> str:
    preferred = {
        "sbp": ["Ave_SBP", "ave_sbp", "SBP", "sbp"],
        "dbp": ["Ave_DBP", "ave_dbp", "DBP", "dbp"],
    }
    for c in preferred[kind]:
        if c in df.columns:
            return c

    probe = [c for c in df.columns if kind.upper() in c.upper()]
    if probe:
        return probe[0]

    raise KeyError(f"Could not find {kind.upper()} column in merged data.")

model_df = merged_df.copy()
sbp_col = find_bp_column(model_df, "sbp")
dbp_col = find_bp_column(model_df, "dbp")

model_df["Hypertension"] = ((model_df[sbp_col] >= 130) | (model_df[dbp_col] >= 80)).astype(int)

if "height" in model_df.columns and "weight" in model_df.columns:
    model_df["BMI"] = model_df["weight"] / ((model_df["height"] / 100.0) ** 2)

special_na_codes = {
    "ever_smk": [99],
    "con_alcohol": [9],
    "drnk_30days": [9],
    "current_smoking": [99],
    "binge_drinking": [9, 99],
}
for col, codes in special_na_codes.items():
    if col in model_df.columns:
        model_df[col] = model_df[col].replace(codes, np.nan)

# Guardrail for obvious out-of-range smoking codes.
if "current_smoking" in model_df.columns:
    model_df.loc[model_df["current_smoking"] > 10, "current_smoking"] = np.nan

def map_alcohol_level(row):
    status = row.get("alcohol_status", np.nan)
    binge = row.get("binge_drinking", np.nan)
    if pd.isna(status):
        return np.nan
    if status == 0:
        return 0
    if status == 2:
        return 1
    if status == 1:
        return 3 if binge == 1 else 2
    return np.nan

def map_smoking_level(row):
    status = row.get("smoke_status", np.nan)
    current = row.get("current_smoking", np.nan)
    if pd.isna(status):
        return np.nan
    if status == 0:
        return 0
    if status == 2:
        return 1
    if status == 1:
        return 3 if current == 3 else 2
    return np.nan

if "alcohol_status" in model_df.columns:
    model_df["alcohol_level"] = model_df.apply(map_alcohol_level, axis=1)
if "smoke_status" in model_df.columns:
    model_df["smoking_level"] = model_df.apply(map_smoking_level, axis=1)

drop_candidates = [
    sbp_col, dbp_col,
    "height", "weight",
    "enns_year", "hhnum", "member_code",
    "regcode", "provcode", "psc", "psurec", "csc", "rhc", "strrec",
    "ms_psucode", "rep_natl", "rep_prov", "wrkplace", "interview_status", "intdate", "enumcode",
    "finalwgt1", "finalwgt4", "wgts", "cu",
    "alcohol", "con_alcohol", "drnk_30days", "drnk_30d_num", "alcohol_status",
    "binge_drinking", "current_smoking", "ever_smk", "smoke_status",
]
model_df = model_df.drop(columns=[c for c in drop_candidates if c in model_df.columns], errors="ignore")

TARGET_COL = "Hypertension"
model_df = model_df.dropna(subset=[TARGET_COL]).copy()

print(f"Model dataframe shape: {model_df.shape}")
print("Target distribution:")
print(model_df[TARGET_COL].value_counts(normalize=True).rename("ratio").to_string())

Model dataframe shape: (151189, 43)
Target distribution:
Hypertension
0    0.663838
1    0.336162


In [5]:
X = model_df.drop(columns=[TARGET_COL]).copy()
y = model_df[TARGET_COL].astype(int).copy()

categorical_cols = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
X = pd.get_dummies(X, columns=categorical_cols, drop_first=False, dummy_na=True)
X = X.replace([np.inf, -np.inf], np.nan)

X_train_raw, X_temp_raw, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=RANDOM_SEED, stratify=y
)
X_cal_raw, X_test_raw, y_cal, y_test = train_test_split(
    X_temp_raw, y_temp, test_size=0.50, random_state=RANDOM_SEED, stratify=y_temp
)

num_cols = X_train_raw.select_dtypes(include=[np.number]).columns.tolist()
train_medians = X_train_raw[num_cols].median()

scaler = StandardScaler()

X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train_raw[num_cols].fillna(train_medians)),
    columns=num_cols, index=X_train_raw.index
)
X_cal_scaled = pd.DataFrame(
    scaler.transform(X_cal_raw[num_cols].fillna(train_medians)),
    columns=num_cols, index=X_cal_raw.index
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test_raw[num_cols].fillna(train_medians)),
    columns=num_cols, index=X_test_raw.index
)

# Restore NaN masks so KNN imputation is still applied after scaling.
X_train_scaled = X_train_scaled.mask(X_train_raw[num_cols].isna())
X_cal_scaled = X_cal_scaled.mask(X_cal_raw[num_cols].isna())
X_test_scaled = X_test_scaled.mask(X_test_raw[num_cols].isna())

print(f"Encoded feature count: {X.shape[1]}")
print(f"Train shape: {X_train_scaled.shape}")
print(f"Calibration shape: {X_cal_scaled.shape}")
print(f"Test shape: {X_test_scaled.shape}")
print(
    f"NaNs before imputation -> train: {X_train_scaled.isna().sum().sum()}, "
    f"cal: {X_cal_scaled.isna().sum().sum()}, test: {X_test_scaled.isna().sum().sum()}"
)

Encoded feature count: 42
Train shape: (105832, 42)
Calibration shape: (22678, 42)
Test shape: (22679, 42)
NaNs before imputation -> train: 3183142, cal: 678924, test: 679622


In [6]:
k_neighbors = 7
imputer = KNNImputer(n_neighbors=k_neighbors)

X_train_standardized = pd.DataFrame(
    imputer.fit_transform(X_train_scaled), columns=X_train_scaled.columns, index=X_train_scaled.index
)
X_cal_standardized = pd.DataFrame(
    imputer.transform(X_cal_scaled), columns=X_cal_scaled.columns, index=X_cal_scaled.index
)
X_test_standardized = pd.DataFrame(
    imputer.transform(X_test_scaled), columns=X_test_scaled.columns, index=X_test_scaled.index
)

print(f"KNN k: {k_neighbors}")
print(
    f"NaNs after imputation -> train: {X_train_standardized.isna().sum().sum()}, "
    f"cal: {X_cal_standardized.isna().sum().sum()}, test: {X_test_standardized.isna().sum().sum()}"
)

prep_bundle = {
    "random_seed": RANDOM_SEED,
    "scaler": scaler,
    "imputer": imputer,
    "k_neighbors": k_neighbors,
    "train_medians": train_medians,
    "numeric_columns": num_cols,
    "feature_columns": X_train_standardized.columns.tolist(),
}
prep_path = ARTIFACT_DIR / "preprocessing_bundle_2015.joblib"
joblib.dump(prep_bundle, prep_path)
print(f"Saved preprocessing bundle: {prep_path}")

KNN k: 7
NaNs after imputation -> train: 0, cal: 0, test: 0
Saved preprocessing bundle: c:\Jon\College\Thesis\V2.2.1.1\main_2015_training_artifacts\preprocessing_bundle_2015.joblib


In [7]:
from copy import deepcopy

from sklearn.ensemble import (
    AdaBoostClassifier,
    ExtraTreesClassifier,
    GradientBoostingClassifier,
    HistGradientBoostingClassifier,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.impute import KNNImputer

from imblearn.over_sampling import ADASYN, SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline


def safe_predict_proba(model, X_df: pd.DataFrame) -> np.ndarray:
    if hasattr(model, "predict_proba"):
        p = model.predict_proba(X_df)
        return p[:, 1] if p.ndim == 2 else p
    if hasattr(model, "decision_function"):
        d = np.asarray(model.decision_function(X_df), dtype=float)
        return 1.0 / (1.0 + np.exp(-d))
    return np.asarray(model.predict(X_df), dtype=float)


def metric_pack(y_true, y_prob, threshold: float = 0.5):
    y_pred = (y_prob >= threshold).astype(int)
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "auc": roc_auc_score(y_true, y_prob),
        "logloss": log_loss(y_true, np.clip(y_prob, 1e-6, 1 - 1e-6)),
    }


neg_pos_ratio = float((y_train == 0).sum() / max(1, (y_train == 1).sum()))


def build_model_configs() -> dict:
    configs = {
        "LogReg": {
            "factory": lambda: LogisticRegression(max_iter=3000, solver="lbfgs", random_state=RANDOM_SEED),
            "param_grid": {"C": [0.5, 1.0]},
            "supports_class_weight": True,
            "weight_mode": "class_weight",
        },
        "RandomForest": {
            "factory": lambda: RandomForestClassifier(
                random_state=RANDOM_SEED,
                n_jobs=-1,
            ),
            "param_grid": {
                "n_estimators": [300],
                "max_depth": [None, 14],
                "min_samples_leaf": [1],
            },
            "supports_class_weight": True,
            "weight_mode": "class_weight_subsample",
        },
        "ExtraTrees": {
            "factory": lambda: ExtraTreesClassifier(
                random_state=RANDOM_SEED,
                n_jobs=-1,
            ),
            "param_grid": {
                "n_estimators": [300],
                "max_depth": [None, 14],
                "min_samples_leaf": [1],
            },
            "supports_class_weight": True,
            "weight_mode": "class_weight",
        },
        "AdaBoost": {
            "factory": lambda: AdaBoostClassifier(random_state=RANDOM_SEED),
            "param_grid": {
                "n_estimators": [250],
                "learning_rate": [0.05, 0.1],
            },
            "supports_class_weight": False,
            "weight_mode": None,
        },
    }

    try:
        from xgboost import XGBClassifier

        configs["XGBoost"] = {
            "factory": lambda: XGBClassifier(
                objective="binary:logistic",
                eval_metric="logloss",
                random_state=RANDOM_SEED,
                n_jobs=-1,
                verbosity=0,
            ),
            "param_grid": {
                "n_estimators": [300],
                "max_depth": [4],
                "learning_rate": [0.05, 0.1],
                "subsample": [0.85],
                "colsample_bytree": [0.85],
            },
            "supports_class_weight": True,
            "weight_mode": "scale_pos_weight",
        }
    except Exception:
        print("XGBoost unavailable -> using GradientBoosting fallback.")
        configs["GradientBoosting"] = {
            "factory": lambda: GradientBoostingClassifier(random_state=RANDOM_SEED),
            "param_grid": {
                "n_estimators": [250],
                "learning_rate": [0.05, 0.1],
                "max_depth": [2],
            },
            "supports_class_weight": False,
            "weight_mode": None,
        }

    try:
        from lightgbm import LGBMClassifier

        configs["LightGBM"] = {
            "factory": lambda: LGBMClassifier(
                random_state=RANDOM_SEED,
                n_jobs=-1,
                verbose=-1,
            ),
            "param_grid": {
                "n_estimators": [300],
                "learning_rate": [0.05, 0.1],
                "num_leaves": [31],
            },
            "supports_class_weight": True,
            "weight_mode": "class_weight",
        }
    except Exception:
        print("LightGBM unavailable -> using HistGradientBoosting fallback.")
        configs["HistGradientBoosting"] = {
            "factory": lambda: HistGradientBoostingClassifier(random_state=RANDOM_SEED),
            "param_grid": {
                "max_depth": [None],
                "learning_rate": [0.05, 0.1],
                "max_iter": [250],
            },
            "supports_class_weight": False,
            "weight_mode": None,
        }

    try:
        from catboost import CatBoostClassifier

        configs["CatBoost"] = {
            "factory": lambda: CatBoostClassifier(
                loss_function="Logloss",
                random_seed=RANDOM_SEED,
                verbose=0,
            ),
            "param_grid": {
                "iterations": [300],
                "depth": [6],
                "learning_rate": [0.05, 0.1],
            },
            "supports_class_weight": True,
            "weight_mode": "auto_class_weights",
        }
    except Exception:
        print("CatBoost unavailable -> using second GradientBoosting family fallback.")
        configs["GradientBoostingV2"] = {
            "factory": lambda: GradientBoostingClassifier(random_state=RANDOM_SEED + 1),
            "param_grid": {
                "n_estimators": [280],
                "learning_rate": [0.05, 0.1],
                "max_depth": [2],
            },
            "supports_class_weight": False,
            "weight_mode": None,
        }

    if len(configs) < 7:
        raise RuntimeError(f"Could not build 7 model families. Found: {list(configs.keys())}")

    # Keep exactly 7 model families.
    return dict(list(configs.items())[:7])


model_configs = build_model_configs()
print(f"Model families ({len(model_configs)}): {list(model_configs.keys())}")

strategy_specs = [
    {"name": "base", "sampler": None, "use_class_weight": False},
    {"name": "smote", "sampler": "smote", "use_class_weight": False},
    {"name": "adasyn", "sampler": "adasyn", "use_class_weight": False},
    {"name": "class_weight", "sampler": None, "use_class_weight": True},
    {"name": "smote_class_weight", "sampler": "smote", "use_class_weight": True},
    {"name": "adasyn_class_weight", "sampler": "adasyn", "use_class_weight": True},
]


def make_sampler(name: str):
    if name == "smote":
        return SMOTE(random_state=RANDOM_SEED, k_neighbors=5)
    if name == "adasyn":
        return ADASYN(random_state=RANDOM_SEED, n_neighbors=5)
    return None


def make_estimator(config: dict, use_class_weight: bool):
    est = deepcopy(config["factory"]())
    if not use_class_weight or not config.get("supports_class_weight", False):
        return est

    mode = config.get("weight_mode")
    if mode == "class_weight":
        est.set_params(class_weight="balanced")
    elif mode == "class_weight_subsample":
        est.set_params(class_weight="balanced_subsample")
    elif mode == "scale_pos_weight":
        est.set_params(scale_pos_weight=neg_pos_ratio)
    elif mode == "auto_class_weights":
        est.set_params(auto_class_weights="Balanced")
    return est


# Strict CV setup: minimum 30 folds and maximum 100 folds for each of the 42 searches.
min_cv_folds = 30
max_cv_folds = 100
requested_cv_folds = 30
if requested_cv_folds < min_cv_folds or requested_cv_folds > max_cv_folds:
    raise ValueError(
        f"requested_cv_folds must be between {min_cv_folds} and {max_cv_folds}, got {requested_cv_folds}."
    )

# Ensure standardized matrices exist even if Cell 7 was not run in this kernel session.
if "X_train_standardized" not in globals():
    required_scaled = ["X_train_scaled", "X_cal_scaled", "X_test_scaled"]
    if not all(name in globals() for name in required_scaled):
        raise NameError(
            "X_train_standardized is missing and scaled matrices are unavailable. Run Cell 6 and Cell 7 first."
        )

    recovered_imputer = None
    if "imputer" in globals():
        recovered_imputer = imputer
    else:
        prep_path = ARTIFACT_DIR / "preprocessing_bundle_2015.joblib"
        if prep_path.exists():
            prep_bundle = joblib.load(prep_path)
            recovered_imputer = prep_bundle.get("imputer")

    if recovered_imputer is None:
        recovered_imputer = KNNImputer(n_neighbors=7)

    try:
        X_train_standardized = pd.DataFrame(
            recovered_imputer.transform(X_train_scaled),
            columns=X_train_scaled.columns,
            index=X_train_scaled.index,
        )
    except Exception:
        X_train_standardized = pd.DataFrame(
            recovered_imputer.fit_transform(X_train_scaled),
            columns=X_train_scaled.columns,
            index=X_train_scaled.index,
        )

    X_cal_standardized = pd.DataFrame(
        recovered_imputer.transform(X_cal_scaled),
        columns=X_cal_scaled.columns,
        index=X_cal_scaled.index,
    )
    X_test_standardized = pd.DataFrame(
        recovered_imputer.transform(X_test_scaled),
        columns=X_test_scaled.columns,
        index=X_test_scaled.index,
    )
    imputer = recovered_imputer
    print("Recovered standardized matrices inside Cell 8.")

# Large sample speed-up for CV; best configuration is refit on full training set.
gridsearch_max_rows = 50000
if len(X_train_standardized) > gridsearch_max_rows:
    frac = gridsearch_max_rows / len(y_train)
    sampled_idx = y_train.groupby(y_train).sample(frac=frac, random_state=RANDOM_SEED).index
    X_search = X_train_standardized.loc[sampled_idx].copy()
    y_search = y_train.loc[sampled_idx].copy()
else:
    X_search = X_train_standardized.copy()
    y_search = y_train.copy()

minority_count_search = int(y_search.value_counts().min())
if minority_count_search < min_cv_folds:
    raise ValueError(
        f"Minority-class count in search data ({minority_count_search}) is below minimum folds ({min_cv_folds})."
    )

effective_cv_folds = min(requested_cv_folds, max_cv_folds, minority_count_search)
cv = StratifiedKFold(n_splits=effective_cv_folds, shuffle=True, random_state=RANDOM_SEED)

print(f"GridSearch training rows: {len(X_search):,} / {len(X_train_standardized):,}")
print(
    f"Strict CV folds per gridsearch: {effective_cv_folds} "
    f"(requested={requested_cv_folds}, cap={max_cv_folds})"
)

search_rows = []
for model_name, config in model_configs.items():
    for spec in strategy_specs:
        strategy_name = spec["name"]
        use_class_weight = spec["use_class_weight"]
        sampler = make_sampler(spec["sampler"])

        estimator = make_estimator(config, use_class_weight=use_class_weight)
        steps = []
        if sampler is not None:
            steps.append(("sampler", sampler))
        steps.append(("model", estimator))
        pipe = ImbPipeline(steps=steps)

        param_grid = {f"model__{k}": v for k, v in config["param_grid"].items()}

        print(f"GridSearch -> {model_name} | {strategy_name} | folds={effective_cv_folds}")
        gs = GridSearchCV(
            estimator=pipe,
            param_grid=param_grid,
            scoring={"f1": "f1", "roc_auc": "roc_auc"},
            refit="f1",
            cv=cv,
            n_jobs=-1,
            verbose=0,
            error_score="raise",
        )

        gs.fit(X_search, y_search)

        # Validate actual splits used by this specific gridsearch run.
        actual_cv_splits = int(getattr(gs, "n_splits_", effective_cv_folds))
        if actual_cv_splits < min_cv_folds:
            raise RuntimeError(
                f"{model_name}/{strategy_name} used {actual_cv_splits} folds, below minimum {min_cv_folds}."
            )

        best_pipe = gs.best_estimator_

        # Refit best model/strategy on full training set before calibration evaluation.
        best_pipe.fit(X_train_standardized, y_train)

        p_cal = np.clip(safe_predict_proba(best_pipe, X_cal_standardized), 1e-6, 1 - 1e-6)
        cal_m = metric_pack(y_cal, p_cal, threshold=0.5)

        model_path = MODEL_DIR / f"{model_name}__{strategy_name}__gridsearch_best.joblib"
        joblib.dump(best_pipe, model_path)

        cv_auc = np.nan
        if "mean_test_roc_auc" in gs.cv_results_:
            cv_auc = float(gs.cv_results_["mean_test_roc_auc"][gs.best_index_])

        search_rows.append(
            {
                "model_name": model_name,
                "strategy": strategy_name,
                "use_class_weight": bool(use_class_weight),
                "sampler": spec["sampler"] if spec["sampler"] is not None else "none",
                "cv_requested_folds": int(requested_cv_folds),
                "cv_folds_used": int(actual_cv_splits),
                "cv_best_f1": float(gs.best_score_),
                "cv_best_auc": cv_auc,
                "cal_accuracy": cal_m["accuracy"],
                "cal_precision": cal_m["precision"],
                "cal_recall": cal_m["recall"],
                "cal_f1": cal_m["f1"],
                "cal_auc": cal_m["auc"],
                "cal_logloss": cal_m["logloss"],
                "best_params": json.dumps(gs.best_params_, sort_keys=True),
                "model_path": str(model_path),
            }
        )

all_gridsearch_df = pd.DataFrame(search_rows)
expected_runs = len(model_configs) * len(strategy_specs)
print(f"Completed gridsearch runs: {len(all_gridsearch_df)} / {expected_runs}")

if all_gridsearch_df["cv_folds_used"].min() < min_cv_folds:
    raise RuntimeError(
        f"Fold audit failed: minimum observed folds {all_gridsearch_df['cv_folds_used'].min()} is below {min_cv_folds}."
    )

print(
    f"CV fold audit passed: min={all_gridsearch_df['cv_folds_used'].min()} "
    f"max={all_gridsearch_df['cv_folds_used'].max()}"
)

all_gridsearch_df = all_gridsearch_df.sort_values(
    ["model_name", "cal_f1", "cal_auc"],
    ascending=[True, False, False],
).reset_index(drop=True)

all_gridsearch_path = ARTIFACT_DIR / "all_42_gridsearch_results_2015_main.csv"
all_gridsearch_df.to_csv(all_gridsearch_path, index=False)

selected_rows = []
for model_name, grp in all_gridsearch_df.groupby("model_name", sort=False):
    by_f1 = grp.sort_values(["cal_f1", "cal_auc"], ascending=False).head(1)
    by_auc = grp.sort_values(["cal_auc", "cal_f1"], ascending=False).head(1)

    picks = (
        pd.concat([by_f1, by_auc], ignore_index=True)
        .drop_duplicates(subset=["model_name", "strategy"])
        .reset_index(drop=True)
    )

    if len(picks) < 2:
        picks = grp.sort_values(["cal_f1", "cal_auc"], ascending=False).head(2).reset_index(drop=True)

    picks["selected_reason"] = ["best_f1", "best_auc"][: len(picks)]
    selected_rows.append(picks)

selected_candidates_df = pd.concat(selected_rows, ignore_index=True)
selected_candidates_df = selected_candidates_df.sort_values(
    ["model_name", "cal_f1", "cal_auc"],
    ascending=[True, False, False],
).reset_index(drop=True)

selected_top2_path = ARTIFACT_DIR / "top2_per_model_from_gridsearch_2015_main.csv"
selected_candidates_df.to_csv(selected_top2_path, index=False)

print("\nTop-2 per model family (for calibration):")
print(selected_candidates_df[["model_name", "strategy", "cal_f1", "cal_auc", "selected_reason"]].to_string(index=False))
print(f"Saved: {all_gridsearch_path}")
print(f"Saved: {selected_top2_path}")

if len(selected_candidates_df) != 14:
    print(
        f"Warning: expected 14 selected candidates (7 models x 2), got {len(selected_candidates_df)}. "
        "Calibration cell will still proceed with available candidates."
    )

Model families (7): ['LogReg', 'RandomForest', 'ExtraTrees', 'AdaBoost', 'XGBoost', 'LightGBM', 'CatBoost']
GridSearch training rows: 50,000 / 105,832
Strict CV folds per gridsearch: 30 (requested=30, cap=100)
GridSearch -> LogReg | base | folds=30
GridSearch -> LogReg | smote | folds=30
GridSearch -> LogReg | adasyn | folds=30
GridSearch -> LogReg | class_weight | folds=30
GridSearch -> LogReg | smote_class_weight | folds=30
GridSearch -> LogReg | adasyn_class_weight | folds=30
GridSearch -> RandomForest | base | folds=30
GridSearch -> RandomForest | smote | folds=30
GridSearch -> RandomForest | adasyn | folds=30
GridSearch -> RandomForest | class_weight | folds=30
GridSearch -> RandomForest | smote_class_weight | folds=30
GridSearch -> RandomForest | adasyn_class_weight | folds=30
GridSearch -> ExtraTrees | base | folds=30
GridSearch -> ExtraTrees | smote | folds=30
GridSearch -> ExtraTrees | adasyn | folds=30
GridSearch -> ExtraTrees | class_weight | folds=30
GridSearch -> ExtraTree

In [8]:
from copy import deepcopy

from sklearn.calibration import CalibratedClassifierCV

try:
    from sklearn.frozen import FrozenEstimator
except Exception:
    FrozenEstimator = None


def expected_calibration_error(y_true, y_prob, n_bins: int = 15) -> float:
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    idx = np.digitize(y_prob, bins) - 1

    ece = 0.0
    for i in range(n_bins):
        m = idx == i
        if np.any(m):
            conf = y_prob[m].mean()
            acc = y_true[m].mean()
            ece += m.mean() * abs(acc - conf)
    return float(ece)


def build_calibrator(fitted_estimator, method: str):
    if FrozenEstimator is not None:
        # Preferred path for newer scikit-learn where cv='prefit' was removed.
        return CalibratedClassifierCV(estimator=FrozenEstimator(fitted_estimator), method=method)

    # Compatibility fallback for older/newer environments if FrozenEstimator is unavailable.
    return CalibratedClassifierCV(estimator=deepcopy(fitted_estimator), method=method, cv=3)


if "selected_candidates_df" not in globals() or selected_candidates_df.empty:
    raise RuntimeError("No selected candidates found. Run Cell 8 first.")

X_cal_fit, X_cal_eval, y_cal_fit, y_cal_eval = train_test_split(
    X_cal_standardized,
    y_cal,
    test_size=0.5,
    random_state=RANDOM_SEED,
    stratify=y_cal,
)

method_rows = []
chosen_rows = []

for _, row in selected_candidates_df.iterrows():
    model_name = row["model_name"]
    strategy = row["strategy"]
    base_model_path = row["model_path"]

    base_model = joblib.load(base_model_path)

    method_fit_objects = {}
    local_rows = []

    for method in ["sigmoid", "isotonic"]:
        calibrator = build_calibrator(base_model, method=method)
        calibrator.fit(X_cal_fit, y_cal_fit)

        p_eval = np.clip(calibrator.predict_proba(X_cal_eval)[:, 1], 1e-6, 1 - 1e-6)
        eval_m = metric_pack(y_cal_eval, p_eval, threshold=0.5)

        local_row = {
            "model_name": model_name,
            "strategy": strategy,
            "base_model_path": base_model_path,
            "calibration_method": method,
            "eval_accuracy": eval_m["accuracy"],
            "eval_precision": eval_m["precision"],
            "eval_recall": eval_m["recall"],
            "eval_f1": eval_m["f1"],
            "eval_auc": eval_m["auc"],
            "eval_logloss": eval_m["logloss"],
            "eval_ece": expected_calibration_error(y_cal_eval, p_eval),
        }
        local_rows.append(local_row)
        method_rows.append(local_row)
        method_fit_objects[method] = calibrator

    local_df = pd.DataFrame(local_rows).sort_values(
        ["eval_recall", "eval_accuracy", "eval_f1", "eval_auc"],
        ascending=False,
    ).reset_index(drop=True)

    best_local = local_df.iloc[0].to_dict()
    best_method = best_local["calibration_method"]

    calibrated_model_path = MODEL_DIR / f"{model_name}__{strategy}__calibrated_{best_method}.joblib"
    joblib.dump(method_fit_objects[best_method], calibrated_model_path)

    best_local["calibrated_model_path"] = str(calibrated_model_path)
    chosen_rows.append(best_local)

calibration_method_df = pd.DataFrame(method_rows).sort_values(
    ["eval_recall", "eval_accuracy", "eval_f1", "eval_auc"],
    ascending=False,
).reset_index(drop=True)

calibration_method_path = ARTIFACT_DIR / "all_calibration_method_results_2015_main.csv"
calibration_method_df.to_csv(calibration_method_path, index=False)

calibrated_candidates_df = pd.DataFrame(chosen_rows).sort_values(
    ["eval_recall", "eval_accuracy", "eval_f1", "eval_auc"],
    ascending=False,
).reset_index(drop=True)

calibrated_candidates_path = ARTIFACT_DIR / "calibrated_14_candidates_ranked_2015_main.csv"
calibrated_candidates_df.to_csv(calibrated_candidates_path, index=False)

best_calibrated_row = calibrated_candidates_df.iloc[0]

print("Calibrated candidates (best method per selected model-strategy):")
print(
    calibrated_candidates_df[
        [
            "model_name",
            "strategy",
            "calibration_method",
            "eval_recall",
            "eval_accuracy",
            "eval_f1",
            "eval_auc",
        ]
    ].to_string(index=False)
)
print("\nBest calibrated candidate (priority: recall -> accuracy):")
print(
    best_calibrated_row[
        [
            "model_name",
            "strategy",
            "calibration_method",
            "eval_recall",
            "eval_accuracy",
            "eval_f1",
            "eval_auc",
        ]
    ].to_string()
)
print(f"Saved: {calibration_method_path}")
print(f"Saved: {calibrated_candidates_path}")

Calibrated candidates (best method per selected model-strategy):
  model_name            strategy calibration_method  eval_recall  eval_accuracy  eval_f1  eval_auc
    CatBoost              adasyn            sigmoid     0.707951       0.752447 0.657808  0.818897
    CatBoost        class_weight            sigmoid     0.706114       0.755887 0.660368  0.822200
     XGBoost        class_weight            sigmoid     0.703490       0.755269 0.658965  0.822333
    LightGBM        class_weight            sigmoid     0.702965       0.755622 0.659122  0.821827
    LightGBM adasyn_class_weight            sigmoid     0.699292       0.751742 0.654389  0.817469
    AdaBoost               smote            sigmoid     0.698767       0.751389 0.653898  0.818817
    AdaBoost  smote_class_weight            sigmoid     0.698767       0.751389 0.653898  0.818817
     XGBoost               smote           isotonic     0.695880       0.752095 0.653604  0.819426
RandomForest                base           i

In [9]:
if "best_calibrated_row" not in globals():
    raise RuntimeError("No best calibrated candidate found. Run Cell 9 first.")

if "build_calibrator" not in globals():
    raise RuntimeError("Calibration helper is missing. Re-run Cell 9.")

best_model_name = best_calibrated_row["model_name"]
best_strategy = best_calibrated_row["strategy"]
best_method = best_calibrated_row["calibration_method"]

best_base_model = joblib.load(best_calibrated_row["base_model_path"])

# Refit the winning calibration method on the full calibration split.
final_calibrator = build_calibrator(best_base_model, method=best_method)
final_calibrator.fit(X_cal_standardized, y_cal)

p_test_final = np.clip(final_calibrator.predict_proba(X_test_standardized)[:, 1], 1e-6, 1 - 1e-6)
test_metrics = metric_pack(y_test, p_test_final, threshold=0.5)

test_metrics_row = {
    "best_model": best_model_name,
    "best_strategy": best_strategy,
    "calibration_method": best_method,
    "selection_priority": "highest_recall_then_accuracy",
    "test_accuracy": test_metrics["accuracy"],
    "test_precision": test_metrics["precision"],
    "test_recall": test_metrics["recall"],
    "test_f1": test_metrics["f1"],
    "test_auc": test_metrics["auc"],
    "test_logloss": test_metrics["logloss"],
    "test_ece": expected_calibration_error(y_test, p_test_final),
}

final_model_path = MODEL_DIR / f"final_best_{best_model_name}__{best_strategy}__{best_method}.joblib"
joblib.dump(final_calibrator, final_model_path)

test_metrics_df = pd.DataFrame([test_metrics_row])
test_metrics_path = ARTIFACT_DIR / "best_calibrated_test_metrics_2015_main.csv"
test_metrics_df.to_csv(test_metrics_path, index=False)

summary_payload = {
    **test_metrics_row,
    "base_model_path": str(best_calibrated_row["base_model_path"]),
    "final_calibrated_model_path": str(final_model_path),
}
summary_path = ARTIFACT_DIR / "best_calibrated_summary_2015_main.json"
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary_payload, f, indent=2)

print("Final best calibrated model on test split:")
print(test_metrics_df.to_string(index=False))
print(f"Saved model: {final_model_path}")
print(f"Saved: {test_metrics_path}")
print(f"Saved: {summary_path}")

Final best calibrated model on test split:
best_model best_strategy calibration_method           selection_priority  test_accuracy  test_precision  test_recall  test_f1  test_auc  test_logloss  test_ece
  CatBoost        adasyn            sigmoid highest_recall_then_accuracy       0.757882        0.620795     0.718914 0.666261   0.82824      0.466917  0.013028
Saved model: c:\Jon\College\Thesis\V2.2.1.1\main_2015_training_artifacts\models\final_best_CatBoost__adasyn__sigmoid.joblib
Saved: c:\Jon\College\Thesis\V2.2.1.1\main_2015_training_artifacts\best_calibrated_test_metrics_2015_main.csv
Saved: c:\Jon\College\Thesis\V2.2.1.1\main_2015_training_artifacts\best_calibrated_summary_2015_main.json


In [10]:
try:
    import shap
except Exception as e:
    raise ImportError(
        "shap is required for the explanation step. Install with: pip install shap"
    ) from e

try:
    from lime.lime_tabular import LimeTabularExplainer
except Exception as e:
    raise ImportError(
        "lime is required for the explanation step. Install with: pip install lime"
    ) from e

if "X_cal_scaled" not in globals() or "X_test_scaled" not in globals():
    raise RuntimeError("Run Cells 1 to 6 first so scaled feature matrices are available.")

prep_path = ARTIFACT_DIR / "preprocessing_bundle_2015.joblib"
if not prep_path.exists():
    raise FileNotFoundError(f"Missing preprocessing bundle: {prep_path}")

prep_bundle = joblib.load(prep_path)
imputer_saved = prep_bundle.get("imputer")
if imputer_saved is None:
    raise RuntimeError("Saved preprocessing bundle does not contain a fitted imputer.")

# Rebuild standardized calibration/test features from saved imputer to avoid re-running long KNN fit.
X_cal_standardized = pd.DataFrame(
    imputer_saved.transform(X_cal_scaled),
    columns=X_cal_scaled.columns,
    index=X_cal_scaled.index,
)
X_test_standardized = pd.DataFrame(
    imputer_saved.transform(X_test_scaled),
    columns=X_test_scaled.columns,
    index=X_test_scaled.index,
)

summary_path = ARTIFACT_DIR / "best_calibrated_summary_2015_main.json"
if not summary_path.exists():
    raise FileNotFoundError(f"Missing best-model summary JSON: {summary_path}")

with open(summary_path, "r", encoding="utf-8") as f:
    summary_payload = json.load(f)

final_model_path = Path(summary_payload["final_calibrated_model_path"])
if not final_model_path.exists():
    raise FileNotFoundError(f"Missing final calibrated model: {final_model_path}")

final_calibrator = joblib.load(final_model_path)

explain_dir = ARTIFACT_DIR / "explanations"
lime_dir = explain_dir / "lime"
explain_dir.mkdir(parents=True, exist_ok=True)
lime_dir.mkdir(parents=True, exist_ok=True)

feature_names = X_test_standardized.columns.tolist()


def final_predict_proba(arr_like):
    if isinstance(arr_like, pd.DataFrame):
        x_df = arr_like[feature_names]
    else:
        x_df = pd.DataFrame(arr_like, columns=feature_names)
    return np.asarray(final_calibrator.predict_proba(x_df))


background_n = min(300, len(X_cal_standardized))
explain_n = min(120, len(X_test_standardized))

background_df = X_cal_standardized.sample(n=background_n, random_state=RANDOM_SEED)
explain_df = X_test_standardized.sample(n=explain_n, random_state=RANDOM_SEED)

# SHAP (model-agnostic on calibrated probability for class 1)
shap_explainer = shap.Explainer(
    lambda z: final_predict_proba(z)[:, 1],
    background_df,
)
shap_values = shap_explainer(explain_df, max_evals=(2 * len(feature_names) + 1))

shap_arr = np.asarray(shap_values.values)
if shap_arr.ndim == 3:
    shap_arr = shap_arr[:, :, 1]

shap_importance_df = (
    pd.DataFrame(
        {
            "feature": feature_names,
            "mean_abs_shap": np.abs(shap_arr).mean(axis=0),
        }
    )
    .sort_values("mean_abs_shap", ascending=False)
    .reset_index(drop=True)
)
shap_path = explain_dir / "shap_importance_2015_main.csv"
shap_importance_df.to_csv(shap_path, index=False)

# LIME local explanations for a handful of test instances
lime_explainer = LimeTabularExplainer(
    training_data=background_df.values,
    feature_names=feature_names,
    class_names=["No_Hypertension", "Hypertension"],
    mode="classification",
    random_state=RANDOM_SEED,
    discretize_continuous=True,
)

lime_rows = []
lime_cases = min(5, len(explain_df))
for case_number, (_, row) in enumerate(explain_df.head(lime_cases).iterrows(), start=1):
    exp = lime_explainer.explain_instance(
        data_row=row.values,
        predict_fn=final_predict_proba,
        labels=(1,),
        num_features=min(12, len(feature_names)),
    )
    html_path = lime_dir / f"lime_case_{case_number}.html"
    exp.save_to_file(str(html_path))

    for feature_rule, weight in exp.as_list(label=1):
        lime_rows.append(
            {
                "case_number": case_number,
                "feature_rule": feature_rule,
                "weight": float(weight),
                "abs_weight": float(abs(weight)),
                "html_file": html_path.name,
            }
        )

lime_local_df = pd.DataFrame(lime_rows)
lime_local_path = explain_dir / "lime_local_weights_2015_main.csv"
lime_local_df.to_csv(lime_local_path, index=False)

lime_global_df = (
    lime_local_df.groupby("feature_rule", as_index=False)
    .agg(mean_abs_weight=("abs_weight", "mean"), mean_weight=("weight", "mean"), n_hits=("weight", "count"))
    .sort_values(["mean_abs_weight", "n_hits"], ascending=[False, False])
    .reset_index(drop=True)
)
lime_global_path = explain_dir / "lime_global_importance_2015_main.csv"
lime_global_df.to_csv(lime_global_path, index=False)

print("SHAP top features:")
print(shap_importance_df.head(15).to_string(index=False))
print("\nLIME global feature rules:")
print(lime_global_df.head(15).to_string(index=False))
print(f"Saved: {shap_path}")
print(f"Saved: {lime_local_path}")
print(f"Saved: {lime_global_path}")
print(f"Saved HTML explanations in: {lime_dir}")

SHAP top features:
      feature  mean_abs_shap
          age       0.238972
          sex       0.040677
smoking_level       0.008883
         fg27       0.008584
alcohol_level       0.006775
         fg12       0.005637
          fg5       0.005079
          fg6       0.004924
          fg3       0.002921
         fg25       0.002860
          fg7       0.002804
         fg20       0.002288
   Total_Iron       0.002273
 Total_Energy       0.002169
         fg16       0.002108

LIME global feature rules:
                 feature_rule  mean_abs_weight  mean_weight  n_hits
                 age <= -0.89         0.149552    -0.149552       1
       smoking_level <= -0.22         0.126518     0.126518       1
          -1.04 < sex <= 0.96         0.125299    -0.125299       2
                 sex <= -1.04         0.124841     0.124841       3
         smoking_level > 1.38         0.117675    -0.117675       3
                   age > 0.76         0.085129     0.085129       2
          -0.

In [11]:
print("Artifacts generated:")
for p in sorted(ARTIFACT_DIR.glob("*")):
    if p.is_file():
        print(f"- {p.name}")

print("\nTrained model files:")
for p in sorted(MODEL_DIR.glob("*.joblib")):
    print(f"- {p.name}")

explain_dir = ARTIFACT_DIR / "explanations"
if explain_dir.exists():
    print("\nExplanation artifacts:")
    for p in sorted(explain_dir.rglob("*")):
        if p.is_file():
            rel = p.relative_to(ARTIFACT_DIR)
            print(f"- {rel.as_posix()}")

Artifacts generated:
- all_42_gridsearch_results_2015_main.csv
- all_calibration_method_results_2015_main.csv
- best_calibrated_summary_2015_main.json
- best_calibrated_test_metrics_2015_main.csv
- calibrated_14_candidates_ranked_2015_main.csv
- calibration_results_2015_main.csv
- merged_2015_clinical_dietary.csv
- model_selection_results_2015_main.csv
- preprocessing_bundle_2015.joblib
- top2_per_model_from_gridsearch_2015_main.csv

Trained model files:
- AdaBoost__adasyn__gridsearch_best.joblib
- AdaBoost__adasyn_class_weight__gridsearch_best.joblib
- AdaBoost__base__gridsearch_best.joblib
- AdaBoost__class_weight__gridsearch_best.joblib
- AdaBoost__smote__calibrated_sigmoid.joblib
- AdaBoost__smote__gridsearch_best.joblib
- AdaBoost__smote_class_weight__calibrated_sigmoid.joblib
- AdaBoost__smote_class_weight__gridsearch_best.joblib
- CatBoost.joblib
- CatBoost__adasyn__calibrated_sigmoid.joblib
- CatBoost__adasyn__gridsearch_best.joblib
- CatBoost__adasyn_class_weight__gridsearch_b